# 🚗 Proyecto Final de Máster en IA: Aprendizaje por Refuerzo y Conducción Autónoma

Entrenamos un agente que conduce en el simulador 3D **Duckietown** usando únicamente píxeles de la cámara, con un enfoque de **RL puro** (la política aprende de la imagen a velocidades de rueda). La nota depende del rendimiento en un **mapa oculto** con obstáculos.

### 📋 Fases del proyecto
* **Fase 1: Calentamiento.** Q-Learning tabular desde cero en `FrozenLake-v1`.
* **Fase 2: Baselines (DQN y PPO).** Dos baselines con Stable-Baselines3 (DQN discreto vía wrapper, PPO continuo).
* **Fase 3: Algoritmo avanzado.** SAC, más la depuración del pipeline y la consolidación del modelo final.

### ⚠️ Contrato de evaluación
1. **Observaciones:** forma `(1, 64, 64)` apilada en 4 frames → `(4, 64, 64)`.
2. **Clases personalizadas:** `CustomCNN` y wrappers definidos e importables (`src/`).
3. **Dry-run:** todo debe ejecutarse en un Colab limpio desde `requirements.txt`.

> **Entrega autosuficiente.** El ZIP contiene el código, `requirements.txt`, **`best_agent.zip`** (en la raíz) y `Report.pdf`. La primera celda descomprime el ZIP si hace falta, monta un **venv Python 3.11** e instala el stack. El modelo final entregado es un **PPO con la CNN corregida** (`best_agent.zip`), evaluable sin ningún truco de acción.

## Dependencias y entorno (Colab)

El kernel de Colab es **Python 3.12**, pero el stack (numpy 1.23.5, gym 0.25.2, Duckietown) requiere **Python 3.11**. Se monta un **venv 3.11** y se ejecuta todo con `{PY}` (subprocess). **Ejecutar en orden.**

In [ ]:
# Detección/descompresión ROBUSTA del proyecto (NO se clona GitHub por defecto).
# Si MAML_entrega_final.zip trae una carpeta raíz interna, al descomprimir en
# /content/MAML el proyecto puede quedar en /content/MAML/MAML_entrega_final/.
# Por eso buscamos en /content/MAML, /content y cwd, y también en sus subdirectorios
# de 1er y 2º nivel. Tras descomprimir, se vuelve a buscar.
import os
import zipfile

def _has_project(d):
    return (os.path.isfile(os.path.join(d, "requirements.txt"))
            and os.path.isfile(os.path.join(d, "train.py"))
            and os.path.isfile(os.path.join(d, "eval.py"))
            and os.path.isdir(os.path.join(d, "src")))

def _candidate_dirs():
    """Bases + subdirectorios de 1er y 2º nivel, sin duplicados y preservando orden."""
    bases = ["/content/MAML", "/content", os.getcwd()]
    seen, dirs = set(), []
    for base in bases:
        if not os.path.isdir(base):
            continue
        levels = [base]
        try:
            lvl1 = [os.path.join(base, d) for d in sorted(os.listdir(base))
                    if os.path.isdir(os.path.join(base, d))]
        except OSError:
            lvl1 = []
        levels += lvl1
        for d1 in lvl1:                       # 2º nivel
            try:
                levels += [os.path.join(d1, d) for d in sorted(os.listdir(d1))
                           if os.path.isdir(os.path.join(d1, d))]
            except OSError:
                pass
        for d in levels:
            if d not in seen:
                seen.add(d)
                dirs.append(d)
    return dirs

def _find_project():
    return next((d for d in _candidate_dirs() if _has_project(d)), None)

PROJECT = _find_project()

# Si no se encuentra, intentar descomprimir el ZIP de entrega y volver a buscar.
if PROJECT is None:
    zip_path = next((z for z in ["/content/MAML_entrega_final.zip",
                                 os.path.join(os.getcwd(), "MAML_entrega_final.zip")]
                     if os.path.isfile(z)), None)
    if zip_path is not None:
        os.makedirs("/content/MAML", exist_ok=True)
        print(f"Descomprimiendo {zip_path} en /content/MAML ...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall("/content/MAML")
        PROJECT = _find_project()             # re-búsqueda recursiva (1er y 2º nivel)

if PROJECT is None:
    raise FileNotFoundError(
        "Proyecto no encontrado (requirements.txt, train.py, eval.py, src/) en "
        "/content/MAML, /content, cwd ni sus subdirectorios. Sube/descomprime el ZIP.")

print("PROJECT =", PROJECT)
%cd $PROJECT
!ls

In [ ]:
# a) Python 3.11 + dependencias de sistema (Duckietown headless)
!sudo add-apt-repository -y ppa:deadsnakes/ppa
!sudo apt-get update -qq
!sudo apt-get install -y -qq python3.11 python3.11-venv python3.11-dev xvfb python3-opengl ffmpeg freeglut3-dev

In [ ]:
# b) Crear el venv 3.11 y definir PY (intérprete del venv usado en TODO el notebook)
!python3.11 -m venv /content/venv-maml
PY = "/content/venv-maml/bin/python"
!{PY} -m pip install -U pip wheel setuptools
!{PY} --version

In [ ]:
# c) Instalar el stack EXACTO con el Python 3.11 del venv (NO el kernel 3.12)
%cd $PROJECT
!{PY} -m pip install -r requirements.txt
# gym-duckietown se instala APARTE y SIN deps (sus dependencias ya van en requirements.txt)
!{PY} -m pip install --no-deps "duckietown-gym-daffy @ git+https://github.com/duckietown/gym-duckietown.git@c76a7fec7f739db4f624f40ca83ce99383665558"
# Re-fijar numpy al final por si alguna dependencia lo cambió
!{PY} -m pip install --force-reinstall --no-deps numpy==1.23.5

In [ ]:
# d) Verificación de imports y versiones
!MPLBACKEND=Agg {PY} -c "import numpy, torch, gym, gymnasium, stable_baselines3; import gym_duckietown; print('numpy', numpy.__version__); print('torch', torch.__version__); print('gym', gym.__version__); print('gymnasium', gymnasium.__version__); print('stable_baselines3', stable_baselines3.__version__); print('import gym_duckietown OK')"

## Fase 1: Calentamiento Tabular (Q-Learning)

Q-Learning tabular **desde cero** (sin librerías de Deep RL) sobre `FrozenLake-v1` (8×8, resbaladiza). Se actualiza la tabla $Q$ con la ecuación de Bellman TD(0) y política $\varepsilon$-greedy con decaimiento. La política greedy aprendida alcanza una **tasa de éxito ≈ 0.52** (frente a ≈ 0 del azar).

Implementación completa en **`q_learning_sandbox.py`**. La celda siguiente la ejecuta (imprime métricas y guarda la curva de media móvil). Opcional (~30 s).

In [ ]:
!MPLBACKEND=Agg {PY} q_learning_sandbox.py

## Fases 2 y 3: Conducción Autónoma en Duckietown

Pipeline en módulos importables (contrato: clases definidas y cargables):
* `src/wrappers.py` → `DuckieWrapper` (RGB → recorte → grises → 64×64 → `(1,64,64)`) y `DiscreteActionWrapper` (DQN).
* `src/cnn.py` → `CustomCNN`; `src/envs.py` → `build_vec_env` con **FrameStack(4)** → `(4,64,64)`.
* `train.py` entrena **DQN/PPO (Fase 2)** y **SAC (Fase 3)**; `eval.py` evalúa. El lanzador `scripts/run_training_plan.py` orquesta los experimentos.

**Mapas de entrenamiento permitidos:** `loop_empty`, `udem1`, `zigzag_dists`, `small_loop`, `straight_road`. El mapa oculto `Duckietown-loop_obstacles-v0` se usa **solo para evaluación, nunca para entrenar** (bloqueado por `ValueError`).

Celdas en **dry-run** (muestran los comandos, **no entrenan**):

In [ ]:
# Fase 2 (baselines): PPO y DQN
!{PY} scripts/run_training_plan.py --stage ppo20k --dry-run --eval-after
!{PY} scripts/run_training_plan.py --stage dqn20k --dry-run --eval-after
# Fase 3 (algoritmo avanzado): SAC
!{PY} scripts/run_training_plan.py --stage sac20k --dry-run --eval-after

### Entrenamiento y modelo final

Los entrenamientos largos **no son obligatorios** para la corrección (el profesor solo carga `best_agent.zip`). Están documentados en **`EXPERIMENTS.md`** y el **`Report.pdf`**.

El modelo final es un **PPO de RL puro** entrenado en `loop_empty` (50k + fine-tuning a 100k) con `action_mode=wheels`, tras corregir una **doble normalización de la CNN** (`src/cnn.py`) que limitaba el aprendizaje. Se entrega como **`best_agent.zip`**.

## Resultados finales (mapa oculto `loop_obstacles`)

Evaluación con `eval.py` (`--init-order model-first`). Recompensa media en el mapa oculto; «mejor rollout» = best-of-10 en vídeo.

| modelo | loop_obstacles (media ± std) | mejor rollout | decisión |
|---|---|---|---|
| baseline antiguo (`wheels_fixed`) | −869.9 ± 548.6 | — | fallback inicial |
| PPO 20k | −924.2 ± 441.4 | — | no supera baseline |
| DQN 20k | −1004.6 ± 30.0 | — | descartado |
| SAC 20k | −1021.7 ± 81.4 | — | no supera baseline |
| **PPO CNN-fix 100k (ft)** | **−487.6 ± 1049.9** | **1249.5** | **MODELO FINAL** |

El modelo final (`best_agent.zip`) mejora la media del baseline antiguo y produce varios episodios completos con recompensa positiva (mejor rollout **1249.5**, 1501 frames). `loop_obstacles` se usó **solo para evaluación**.

## Evaluación del modelo final

La celda copia `best_agent.zip` (raíz del ZIP) a `models/` y lo evalúa en el **mapa oculto** `loop_obstacles` (`--allow-eval`, solo evaluación). Se carga con `action_mode=wheels` por defecto (sin truco).

In [ ]:
# Preparar best_agent.zip (viene en la raíz del ZIP de entrega)
%cd $PROJECT
import os, shutil
os.makedirs("models", exist_ok=True)
if os.path.exists("best_agent.zip"):
    shutil.copy("best_agent.zip", "models/best_agent.zip")
    print("OK: best_agent.zip -> models/best_agent.zip")
elif not os.path.exists("models/best_agent.zip"):
    raise FileNotFoundError("No se encontró best_agent.zip (raíz o models/).")
!ls -lh models/best_agent.zip

In [ ]:
# Evaluación en el mapa OCULTO loop_obstacles (SOLO evaluación: --allow-eval)
!env MPLBACKEND=Agg CUDA_VISIBLE_DEVICES="" xvfb-run -a {PY} eval.py --algo ppo --model models/best_agent --map Duckietown-loop_obstacles-v0 --episodes 5 --allow-eval --device cpu --init-order model-first --seed 42

## 🎥 Vídeo final del agente conduciendo (mapa oculto)

**Lo más importante de la entrega.** Genera un MP4 del modelo final conduciendo en el **mapa oculto `loop_obstacles`** con la **cámara real del robot** (`--video-source wrapper_rgb`). Prueba 10 rollouts y guarda el de **mayor recompensa** (best-of-10), evitando mostrar un episodio degenerado. El vídeo se **reproduce embebido inline en Colab/Jupyter** (celda siguiente), sin depender de rutas externas.

In [ ]:
# VÍDEO FINAL — modelo conduciendo en loop_obstacles (NO entrena; carga best_agent.zip).
%cd $PROJECT
import base64, os
from IPython.display import HTML, Video, display

VIDEO_OUT = "outputs/best_agent_loop_obstacles.mp4"
!env MPLBACKEND=Agg CUDA_VISIBLE_DEVICES="" xvfb-run -a {PY} scripts/make_eval_video.py --algo ppo --model models/best_agent --map Duckietown-loop_obstacles-v0 --allow-eval --video-source wrapper_rgb --rollouts 10 --out {VIDEO_OUT} --device cpu --init-order model-first --seed 42

# Reproducir el vídeo EMBEBIDO inline en Colab/Jupyter (base64; no depende de rutas).
if os.path.exists(VIDEO_OUT):
    try:
        display(Video(VIDEO_OUT, embed=True, width=480))           # reproductor inline en Colab
    except Exception:
        _b64 = base64.b64encode(open(VIDEO_OUT, "rb").read()).decode()
        display(HTML(f'<video width="480" controls>'
                     f'<source src="data:video/mp4;base64,{_b64}" type="video/mp4"></video>'))
    print("OK: vídeo embebido |", VIDEO_OUT)
else:
    print("No se generó el vídeo; revisa la salida de make_eval_video.py más arriba.")

---

## 📦 Requisitos de entrega y evaluación

El profesor cargará **`best_agent.zip`** en su propio script de evaluación, sobre un entorno limpio basado estrictamente en `requirements.txt`, y lo ejecutará en el mapa oculto `Duckietown-loop_obstacles-v0`. Si el modelo no carga «a la primera» (versiones, clases no encontradas, *shape errors*), la nota de la parte práctica es 0.

**Cómo cumple esta entrega:** el ZIP incluye `Challenge_RL.ipynb`, `requirements.txt` (versiones con `==`), `q_learning_sandbox.py`, `train.py`, `eval.py`, `src/`, `scripts/`, **`best_agent.zip`** (raíz) y **`Report.pdf`**. Observación `(1,64,64)` → FrameStack(4) → `(4,64,64)`; `CustomCNN` y wrappers en `src/` (importables). El modelo final se entrena y evalúa con `action_mode=wheels` (por defecto), por lo que carga sin ningún truco. `loop_obstacles` solo se usa para evaluación, nunca para entrenar.